# Notebook for transcribing audio using Google Cloud Speech-to-Text

### imports

In [1]:
import quail
import os
import shutil
import pickle
import re
import pandas as pd
from nltk.corpus import stopwords

## create speech context from union of words in annotations

In [2]:
atlep1_df = pd.read_pickle('../../data/annotations_dfs/atlep1.p')
atlep2_df = pd.read_pickle('../../data/annotations_dfs/atlep2.p')
arrdev_df = pd.read_pickle('../../data/annotations_dfs/arrdev.p')

In [3]:
def meets_word_criteria(string):
    """
    Removes words with characters not wanted in auto transcriber speech context
    """
    
    good_word = True
    
    # remove words containing digits
    if any(char.isdigit() for char in string):
        good_word = False
    
    # remove words surrounded by single quotes and possessives (avoid duplicates in nested quotations & possessives)
    if string.startswith("'") or string.endswith("'") or string.endswith("'s"):
        good_word = False
        
    # remove unhelpful simple words
    if len(string) <= 2:
        good_word = False
    
    return good_word

In [4]:
def create_speech_context(df):
    """
    Creates episode-specific speech context from video annotations
    """
    
    # use Narrative details (internal and external), Characters on screen, Speech, Character speaking, and Setting
    word_cols = [df.columns[i] for i in [2,3,4,6,7,9]]
    
    # create single string of all text
    allwords = ' '.join(df.loc[:,word_cols].apply(lambda x: ' '.join(x.dropna()), axis=1).values.tolist())
    
    # remove all characters except spaces (catches \n and \t), letters, apostrophes, dashes
    no_punctuation = re.sub("[^\w\s'-]+", '', allwords)
    
    speech_context = []
    
    # split words into list
    for word in no_punctuation.split():
        # identify unique words that meet criteria
        if word.upper() not in speech_context and meets_word_criteria(word):
            speech_context.append(word.upper())

    # remove English stopwords
    speech_context_nostop = [word for word in speech_context if word not in stopwords.words('english')]
    
    return speech_context_nostop

In [5]:
atlep1_speech_context = create_speech_context(atlep1_df)
atlep2_speech_context = create_speech_context(atlep2_df)
arrdev_speech_context = create_speech_context(arrdev_df)

## load in experiment data and mapings between subject ID & PsiTurk ID

In [6]:
with open('../../data/pickles/expdf.p', 'rb') as f:
    expdf = pickle.load(f)

with open('../../data/pickles/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

## set some paths

In [7]:
audiodir = os.path.abspath('../../data/audio/')
transcdir = os.path.abspath('../../data/transcriptions/automatic/')
keypath = os.path.abspath('../../../google-credentials/cloud-speech-credentials.json')

## create directory structure

In [8]:
# for sid, data in id_maps.items():
#     for ses, turkid in data.items():
#         folder = os.path.join(transcdir,sid,turkid)
#         if not os.path.isdir(folder) and not os.path.isdir(os.path.join(transcdir,'drops',sid)):
#             os.makedirs(folder, exist_ok=True)

## transcribe all audio files not previously transcribed

In [9]:
turkids = [tid for l in [list(ses.values()) for ses in id_maps.values()] for tid in l if tid]

done = False

# walk audio folder
for root, dirs, files in os.walk(audiodir):
    
    # ignore parent dirs with hiden files
    if [f for f in files if not f.startswith('.')]:
        # assign psiturk id
        turkid = files[0].split('-')[0]
        # ignore drops
        if turkid in turkids:
            # assign subject id
            sid = expdf.loc[expdf.uniqueid==turkid]['Subject ID'].values[0]
            audio_files = [file for file in files if file.endswith('wav')]
            
            for audio_file in audio_files:
                
                # set correct speech context
                if (any([af.split('-')[1].startswith('prediction') for af in audio_files]) 
                    or audio_file.split('-')[1].startswith('delayed')):
                    speech_context = atlep1_speech_context
                elif 'A' in sid:
                    speech_context = atlep2_speech_context
                else:
                    speech_context = arrdev_speech_context
                
                af_path = os.path.join(root,audio_file)
                save_dir = os.path.join(transcdir,sid,turkid)
                
                # skip over previously decoded audio
                if not os.path.isfile(os.path.join(save_dir,audio_file+'.txt')):
                    
                # decode audio file and save in specified dir
                    print('decoding ' + audio_file)
                    quail.decode_speech(af_path, keypath=keypath, save=True, save_dir=save_dir,
                                        speech_context=speech_context, max_alternatives=5)
                
                else:
                    print('already finished '+ audio_file)

already finished debugvnS2Q:debugG20gK-prediction.wav
already finished debugvnS2Q:debugG20gK-recall.wav
already finished debugRNwge:debugv0fyi-recall.wav
already finished debugRNwge:debugv0fyi-prediction.wav
already finished debugpdN5k:debug3flgN-recall.wav
already finished debugpdN5k:debug3flgN-prediction.wav
already finished debugWvhTI:debug65Thd-recall.wav
already finished debugWvhTI:debug65Thd-prediction.wav
already finished debug3dOrm:debugAjPUS-prediction.wav
already finished debug3dOrm:debugAjPUS-recall.wav
already finished debugfO0us:debug6tlz1-recall.wav
already finished debugfO0us:debug6tlz1-prediction.wav
already finished debugLou4K:debug5KpMQ-prediction.wav
already finished debugLou4K:debug5KpMQ-recall.wav
already finished debugJ2iI9:debugD6tOV-delayed.wav
already finished debugJ2iI9:debugD6tOV-recall.wav
already finished debugNTcAd:debugiaabY-delayed.wav
already finished debugNTcAd:debugiaabY-recall.wav
already finished debuglAycr:debugoFV0D-recall.wav
already finished deb